# Additional solver steps: postprocessing the solution after each update

Many applications have to touch the solution after each time step: They
clip unphysical values such as negative densities, project the solution back
onto a constraint manifold, or enforce algebraic constraints of the PDE.
ExaGRyPE's CCZ4 formulation of the Einstein equations, for example, requires
the conformal metric to have unit determinant and the conformal
extrinsic curvature to be trace-free. Such operations are no part of the PDE
terms. They are additional steps of the solver, which have to run after each
update of the solution.

In this tutorial, we add such a step to the Euler setup of
@ref tutorials_exahype2_10_euler and @ref tutorials_exahype2_12_higher-order-solvers.
We run through the following steps:

- how ExaHyPE describes a time step as a task graph;
- how to add a point-wise postprocessing step to a solver;
- where the step ends up for the Finite Volumes and for Runge-Kutta schemes;
- what happens under the hood, i.e. how you can add arbitrary steps to the
  task graph after it has been created, but before it is compiled.

We assume that you have completed @ref tutorials_exahype2_12_higher-order-solvers,
as we reuse its PDE terms without further explanation.

## Time steps as task graphs

ExaHyPE does not hard-code what happens within a time step. Instead, the
numerical scheme of each solver describes a time step as a task graph
(peano4.api.taskgraph.Graph): Each task is one logical step such as
"reconstruct the patch with its halo" or "update the cell", and the tasks'
dependencies say which step has to finish before another one may start.
The ExaHyPE project hands this graph over to a task graph compiler, which
maps it onto as few mesh traversals as possible. If you run a script with
the logging verbosity TEXT, ExaHyPE prints the task graphs that it hands over
to the compiler.

An additional solver step hence is an additional task. It has to be wired
into the graph such that it reads the new solution once this solution has
been computed, and before anybody else reads it. For the Finite Volumes,
this means after the cell update, but before the solution is projected onto
the faces, where the neighbours pick it up, and before the CFL condition of
the next time step is evaluated.

## Configuring the environment

As in the previous tutorials, we add Peano's Python API to the search path:

In [ ]:
import sys, os

sys.path.insert(0, os.path.abspath("../../src"))

import peano4.api
import exahype2

In [ ]:
try:
    import ipywidgets, matplotlib
except ImportError:
    !{sys.executable} -m pip install ipywidgets matplotlib

## Project, solver and PDE terms

The project and the solver are those of
@ref tutorials_exahype2_12_higher-order-solvers. We start with the Finite
Volumes.

In [ ]:
project = exahype2.api.Project(
    namespace    = ["tutorials", "exahype2", "additionalsteps"],
    project_name = "AdditionalSteps",
    directory    = "20_additional-solver-steps",
    executable   = "additional-steps",
)

In [ ]:
solver = exahype2.api.Solver(
    "EulerFV",
    4,   # unknowns
    0,   # auxiliary variables
)
solver.add_user_solver_includes("#include <cmath>\n#include <algorithm>\n")

solver.add_pde_term(exahype2.api.Flux(
    implementation = '''
      const double irho  = 1.0 / Q[0];
      constexpr double gamma = 1.4;
      const double p = (gamma - 1.0) * (Q[3] - 0.5 * irho * (Q[1] * Q[1] + Q[2] * Q[2]));
      switch (normal) {
        case 0:
          F[0] = Q[1];
          F[1] = irho * Q[1] * Q[1] + p;
          F[2] = irho * Q[1] * Q[2];
          F[3] = irho * Q[1] * (Q[3] + p);
          break;
        case 1:
          F[0] = Q[2];
          F[1] = irho * Q[2] * Q[1];
          F[2] = irho * Q[2] * Q[2] + p;
          F[3] = irho * Q[2] * (Q[3] + p);
          break;
      }
    ''',
    stateless = False,
))

solver.add_pde_term(exahype2.api.MaxEigenvalue(
    implementation = '''
      const double irho  = 1.0 / Q[0];
      constexpr double gamma = 1.4;
      const double p = (gamma - 1.0) * (Q[3] - 0.5 * irho * (Q[1] * Q[1] + Q[2] * Q[2]));
      const double c = std::sqrt(gamma * p * irho);
      const double u = Q[1 + normal] * irho;
      lambdaMax[0] = std::max(std::abs(u + c), std::abs(u - c));
    ''',
    stateless = False,
))

solver.add_pde_term(exahype2.api.InitPointWisely(
    implementation = '''
      constexpr double gamma = 1.4;
      constexpr double p     = 1.0;
      constexpr double u     = 1.0;
      const double     rho   = 1.0 + 0.2 * std::sin(2.0 * M_PI * (x[0] + 0.1) / 0.2);
      Q[0] = rho;
      Q[1] = rho * u;
      Q[2] = 0.0;
      Q[3] = p / (gamma - 1) + 0.5 * rho * u * u;
    ''',
    stateless = False,
))

patch_size  = 9
volume_size = 0.0075

solver.numerical_scheme  = exahype2.api.fv.RusanovWithExplicitEuler(solver, patch_size)
solver.grid_construction = exahype2.api.fv.SetupSimulationOverRegularGrid(solver, volume_size=volume_size)
solver.plotter           = exahype2.api.fv.PlotAsVTKFile(solver, first_plot_time_stamp=0.0, time_in_between_plots=0.05)

## Adding a point-wise postprocessing step

Our additional step is a density floor: Whenever the density in a volume
drops below a minimum density, we raise it to this minimum and keep the
velocity and the pressure. This is a crude version of the positivity
fixes that many codes use to keep densities and pressures positive. To
see an effect, we pick an unrealistically high floor of 0.9. Our density
wave oscillates between 0.8 and 1.2, i.e. the floor cuts off its troughs.

The solver routine add_point_wise_postprocessing() takes a name for the
step and a C++ snippet. ExaHyPE runs the snippet for each volume of each
patch after every update of the solution. Within the snippet, Q points to
the unknowns of one volume, which the snippet may read and write, and x
holds the position of the volume's centre.

In [ ]:
solver.add_point_wise_postprocessing(
    "Density floor",
    '''
      constexpr double gamma      = 1.4;
      constexpr double minDensity = 0.9;
      if (Q[0] < minDensity) {
        const double u = Q[1] / Q[0];
        const double v = Q[2] / Q[0];
        const double p = (gamma - 1.0) * (Q[3] - 0.5 * Q[0] * (u * u + v * v));
        Q[0] = minDensity;
        Q[1] = minDensity * u;
        Q[2] = minDensity * v;
        Q[3] = p / (gamma - 1.0) + 0.5 * minDensity * (u * u + v * v);
      }
    ''',
)

You can call add_point_wise_postprocessing() multiple times. The steps then
run one after the other, in the order in which you have added them.

The remaining steps are the same as in the previous tutorials:

In [ ]:
project.add_solver(solver)

project.set_global_simulation_parameters(
    dimensions   = 2,
    offset       = [-0.1, -0.1],
    size         = [ 0.2,  0.2],
    min_end_time = 0.2,
    max_end_time = 0.2,
    periodic_BC  = [True, True, True],
)

project.set_Peano4_installation(os.path.abspath("../../"), peano4.api.output.CompileMode.RELEASE)

peano4_project = project.generate_Peano4_project(verbosity=peano4.api.LoggerVerbosity.TEXT)
peano4_project.generate()

Among the output, you find the time stepping task graph twice: once as the
numerical scheme has created it, and once with the postprocessing. The
picture below shows the latter:

![Time stepping task graph of the Finite Volumes with the density floor](illustrations/20_additional-solver-steps-fv-task-graph.png)

The task "EulerFV_Density floor after Update cell" sits directly behind the
cell update. All tasks which used to depend on the cell update, i.e. the
projection onto the faces and the evaluation of the CFL condition, now
depend on the density floor. The new task reads and writes cell data only,
so the task graph compiler fuses it into the same mesh traversal as the cell
update: The additional step does not cost an additional mesh traversal.

We build and run the code:

In [ ]:
!cd 20_additional-solver-steps && make -j4

In [ ]:
!cd 20_additional-solver-steps && ./additional-steps

In [ ]:
import tarch.plotter.api.notebook

tarch.plotter.api.notebook.show_solution(
    "20_additional-solver-steps/solution-EulerFV.peano-patch-file",
    unknown       = 0,
    unknown_names = ["rho", "rho u_1", "rho u_2", "E"],
)

After the first time step, the troughs of the wave are flat: The density
never drops below 0.9.

## Runge-Kutta: postprocessing after each stage

A Runge-Kutta scheme does not compute the new solution in one go. It
computes a sequence of intermediate solutions, the stages, and evaluates the
right-hand side of the PDE for each of them
(@ref tutorials_exahype2_12_higher-order-solvers). If we postprocessed the
final solution only, the right-hand side would be evaluated for stages which
violate our constraints. For a density floor, a stage could even carry a
negative density, which the flux cannot handle. The postprocessing therefore
has to run after each stage, too.

You do not have to do anything for this. If we switch to the Finite
Differences with Runge-Kutta time stepping

the very same call of add_point_wise_postprocessing() yields the task graph
below. We use a Runge-Kutta scheme of order two to keep the picture small:

![Time stepping task graph of Runge-Kutta of order two with the density floor](illustrations/20_additional-solver-steps-rk-task-graph.png)

The density floor now runs twice per time step: After "Prepare stage 1",
which computes the input of the second stage, and after
"Complete time step", which computes the new solution. In both cases, it
runs before the outcome is projected onto the faces. The neighbours hence
see the postprocessed data only.

The picture below compares the density along $y=0$ after one period for
both schemes, with and without the floor:

![Density along y=0 after one period with and without density floor](illustrations/20_additional-solver-steps-profile.png)

With Runge-Kutta, the floor cuts off the trough of the wave, while the
rest of the wave remains on top of the exact solution. After one period,
the smallest density is exactly 0.9: The floor holds for every stage, not
only for the final solution of each time step.

The floor adds mass whenever it raises the density, i.e. the scheme is no
longer conservative. For the Finite Volumes, this is particularly visible:
Their numerical diffusion damps the wave such that it stays above the floor
after a few time steps. Until then, however, the floor has added mass, and
the whole profile ends up above the profile without floor. A density floor
hence is a last resort which keeps a code alive, not a physical model.
Constraint enforcement, as we discuss below, is different: It removes errors
which the exact solution would not have in the first place.

## Under the hood: tags and graph surgery

The mechanism behind add_point_wise_postprocessing() is generic. You can use
it to add any task to a solver's task graphs.

**Tags.** Every task (peano4.api.taskgraph.Task) carries a dictionary of
tags. The task graph compilers ignore them. They allow the code which builds
a graph to tell others what a task does, without that code having to know
how the graph is built. A task can carry arbitrarily many tags, one per key.
Each numerical scheme of ExaHyPE tags every task that writes a new solution
with exahype2.api.Solver.SolutionUpdateTaskTag. A Runge-Kutta scheme tags the
tasks which write the input of a further stage, i.e. an intermediate
solution, with exahype2.api.Solver.IntermediateSolutionUpdateTag instead. For
the scheme of order two above, "Prepare stage 1" carries the intermediate tag
and "Complete time step" the solution update tag. The value of either tag tells you the solver,
the data that the task writes and the guard, i.e. the C++ condition which
selects the cells the task updates. If a scheme computes the new solution in
several tasks, as the optimistic Finite Volumes
exahype2.api.fv.RusanovWithOptimisticExplicitEuler do, it tags each of them.

**Graph surgery.** peano4.api.taskgraph.Graph offers find_tasks(tag), which
returns all tasks with a certain tag, and insert_after(anchor, task).
find_tasks() accepts several tags, too, and then returns all tasks which
carry at least one of them. The
latter adds a new task to the graph, makes it depend on anchor, and
redirects all tasks which have depended on anchor so far to the new task.

**The hook.** Once the grid construction, the numerical scheme and the
plotter have built the three task graphs of a solver, i.e. the graphs for
the setup, the time stepping and the plotting, ExaHyPE calls the solver's
routine postprocess_task_graphs(). Only afterwards, it hands the graphs over
to the compiler. The default implementation inserts the point-wise
postprocessing steps. You can override it in a subclass of
exahype2.api.Solver to alter the graphs in any way you like. The sketch
below adds a task with your own action set after each solution update and,
as it searches for both tags, after each Runge-Kutta stage. Drop the
intermediate tag from the list if your step should run once per time step
only:

A few rules keep such steps correct:

- Declare all data that the new task reads or writes via use_cell(),
  use_face() or use_vertex(). The compiler relies on these declarations
  when it decides which tasks it may fuse into one mesh traversal.
- Use the guard of the tag. Otherwise, your task would also touch cells
  that the numerical scheme has not updated, e.g. cells of another solver.
- Insert the task after the anchor rather than adding a plain dependency.
  A plain dependency on the anchor would allow the projection onto the
  faces to run before your task, i.e. the neighbours could see data which
  has not been postprocessed yet.

The numerical schemes have to keep their part of the contract, too: They have
to tag all tasks which write a new solution or an intermediate solution
with the respective tag, and they must not read this
solution within the same task. This is why the Runge-Kutta scheme splits
the computation of each stage into two tasks, "Prepare stage k" and
"Project stage k onto faces": the postprocessing has to fit in between.

## Outlook

In @ref tutorials_exahype2_exagrype_522_single-black-hole, the CCZ4 solver
runs without constraint enforcement. The CCZ4 solver uses the mechanism of
this tutorial to offer it: If you pass enforce_constraints=True to
exagrype.api.CCZ4Solver, it adds exagrype::ccz4::enforceCCZ4constraints() as
point-wise postprocessing. The enforcement then runs after each time step of
the Finite Volumes, and after each Runge-Kutta stage when you switch to the
Finite Differences.